# Stage 07 — H3→H4 Bridge (Exploratory / Secondary) — vFINAL

## Scientific role
This notebook tests whether larger **H3 cue-transport instability** is associated with larger **H4 operational deterioration** across the fixed universe of **8 cue families × 4 BanglaClick platforms = 32 cue×platform cells**.

This is **secondary/exploratory observational corroboration**. It is **not** a new confirmatory hypothesis, does **not** rescue or replace the confirmatory H3 procedure, and does **not** support causal platform claims.

A leave-one-platform-out (LOO) platform-fixed-effect analysis is included only as a **secondary robustness layer**. It does not alter the fixed 32-cell primary universe, the four primary association pairs, or the primary joint-bootstrap analysis.

### Frozen association pairs
1. `|ΔRD|` vs cue-conditioned lift deterioration
2. `|ΔRD|` vs cue-conditioned error-capture deterioration
3. `|ΔNLL-gap|` vs cue-conditioned lift deterioration
4. `|ΔNLL-gap|` vs cue-conditioned error-capture deterioration

H3 uses the frozen Stage-05 definitions: `ΔRD = target − source` cue risk-difference transport and `ΔNLL-gap = target − source` cue-conditioned EBM proper-loss gap, averaged across the five locked EBM seeds.

H4 deterioration is **source − target** (positive means worse target operation).

### Global 10% policy is preserved exactly
For each model×seed×domain, rows are globally ranked once using the Stage-06 scores. The globally reviewed count is `ceil(0.10 × N)` under the ordinary cluster-bootstrap row multiplicities. **No cue-specific re-ranking, quota, or top-k is ever performed.**

For a cue, only globally selected row multiplicity belonging to cue-present rows contributes to that cue's numerator. The H4 denominators remain the original global denominators:

- cue-conditioned lift contribution = `(selected positive cue-present mass / global reviewed mass) / global prevalence`;
- cue-conditioned error-capture contribution = `selected error cue-present mass / global total-error mass`.

The five seeds are **not independent observations**. Within each model, the five seeds are averaged first; the four locked model families are then equally weighted.

### Inference
A **joint 5,000-replicate cluster bootstrap** draws one source resample and one resample for each BanglaClick platform per replicate and reuses those row multiplicities across all models, seeds, cues and association endpoints. Any undefined prespecified component makes that association replicate invalid. Invalid replicates are counted explicitly. No NaNs are silently filtered, no rejection resampling is used, and no cell/dimension is removed post hoc.


## Input contract

Read-only inputs are exactly one canonical provenance chain:

**Stage 05 FINAL → Stage 06 FINAL → Stage 07**

Stage 05 must be the canonical final Stage-05 root containing:
- the three immutable prediction Parquets;
- the corrected/final `H3_RESULTS.json`;
- `STAGE05_ARTIFACT_MANIFEST.json`;
- `STAGE05_SHA256SUMS.txt`;
- `STAGE_STATUS.json`.

Stage 06 must be the independently audited final Stage-06 root containing:
- `H4_RESULTS.json`;
- `REPRODUCIBILITY_MANIFEST.json`;
- `STAGE06_SHA256SUMS.txt`;
- `UPSTREAM_PROVENANCE_GATE.json`;
- `FINAL_STATISTICAL_SUMMARY.json`;
- `BOOTSTRAP_RUNTIME.json`;
- `FIXED_BUDGET_TABLES.csv`;
- `STAGE_STATUS.json`.

Legacy/split-root discovery is disabled. Predictions and H3 must come from the same exact canonical Stage-05 root, and Stage 06 must cryptographically point back to that same Stage-05 identity.

No model loading, transformer inference, calibration fitting, threshold tuning, cue selection, policy tuning, or cue-specific reranking occurs in this stage.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
import hashlib, json, math, os, platform, re, subprocess, sys, time
import numpy as np
import pandas as pd
from scipy.stats import rankdata

STAGE_ID = "07"
STAGE_NAME = "07_H3_H4_BRIDGE_EXPLORATORY_vFINAL"
PROTOCOL_VERSION = "SWARABYANJAN-CROSSPLATFORM-v1.0.0"
BRIDGE_VERSION = "H3-H4-BRIDGE-vFINAL-1.0.1"
SEEDS = [42, 137, 2718, 31415, 65537]
MODEL_KEYS = ["ebm", "tfidf_lr", "banglabert", "xlmr"]
CUE_FAMILIES = [
    "curiosity_information_withholding",
    "direct_address_imperative",
    "exaggeration_superlative",
    "interrogative_structure",
    "punctuation_intensity",
    "sensational_emotional_loading",
    "specificity_numerals",
    "vague_deictic_reference",
]
PLATFORMS = ["news", "facebook", "youtube", "twitter"]
PRIMARY_BUDGET = 0.10
BOOTSTRAP_REPS = 5000
BOOTSTRAP_SEED = 20260912
PARITY_ATOL = 1e-12
PARITY_RTOL = 1e-10

BRANCH_ROOT = Path(os.environ["SWARABYANJAN_BRANCH_ROOT"]).resolve() if os.environ.get("SWARABYANJAN_BRANCH_ROOT") else None
WORK_ROOT = BRANCH_ROOT if BRANCH_ROOT is not None else (Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd())
SEARCH_ROOTS = ([BRANCH_ROOT, Path("/kaggle/input")] if BRANCH_ROOT is not None and Path("/kaggle/input").exists() else [BRANCH_ROOT] if BRANCH_ROOT is not None else ([Path("/kaggle/input"), WORK_ROOT, Path.cwd()] if Path("/kaggle/input").exists() else [WORK_ROOT, Path.cwd()]))
OUT_ROOT = WORK_ROOT / "03_results" / STAGE_NAME
if OUT_ROOT.exists() and any(OUT_ROOT.iterdir()):
    raise RuntimeError(f"Refusing to overwrite non-empty output directory: {OUT_ROOT}")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(path, chunk=8*1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                return h.hexdigest()
            h.update(b)

def _json_builtin(x):
    """Recursively convert numpy/pandas/path scalars to strict JSON built-ins."""
    if x is None or isinstance(x, (str, int, float, bool)):
        if isinstance(x, float) and not np.isfinite(x):
            return None
        return x
    if isinstance(x, np.bool_):
        return bool(x)
    if isinstance(x, np.integer):
        return int(x)
    if isinstance(x, np.floating):
        v = float(x)
        return v if np.isfinite(v) else None
    if isinstance(x, np.ndarray):
        return [_json_builtin(v) for v in x.tolist()]
    if isinstance(x, Path):
        return str(x)
    if isinstance(x, (Counter, defaultdict, dict)):
        return {str(k): _json_builtin(v) for k, v in dict(x).items()}
    if isinstance(x, (list, tuple, set)):
        return [_json_builtin(v) for v in x]
    if x is pd.NA:
        return None
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    raise TypeError(f"Unsupported JSON value type: {type(x).__name__}")

def write_json(path, obj):
    safe = _json_builtin(obj)
    Path(path).write_text(
        json.dumps(safe, ensure_ascii=False, sort_keys=True, indent=2, allow_nan=False) + "\n",
        encoding="utf-8", newline="\n"
    )

def write_text(path, text):
    Path(path).write_text(str(text), encoding="utf-8", newline="\n")

def set_status(status, message, details=None):
    write_json(OUT_ROOT / "STAGE_STATUS.json", {
        "stage_id": STAGE_ID, "bridge_version": BRIDGE_VERSION,
        "status": status, "message": message, "details": details or {},
        "timestamp_utc": utc_now(),
    })
    print(f"STAGE 07 STATUS: {status} — {message}", flush=True)

def hard_stop(message, details=None):
    set_status("HARD_STOP", message, details)
    raise RuntimeError(message)

# Static regression test for the exact failure class from the first real bridge run.
_json_test_path = OUT_ROOT / "_JSON_SERIALIZATION_SELFTEST.json"
_json_test_payload = {
    "python_bool": True,
    "numpy_bool": np.bool_(True),
    "numpy_int": np.int64(7),
    "numpy_float": np.float64(0.25),
    "numpy_array": np.asarray([np.bool_(False), np.bool_(True)]),
    "counter": Counter({"x": np.int64(2)}),
}
write_json(_json_test_path, _json_test_payload)
_json_reload = json.loads(_json_test_path.read_text(encoding="utf-8"))
assert _json_reload["numpy_bool"] is True and _json_reload["numpy_int"] == 7
_json_test_path.unlink()
print("JSON serialization self-test: PASS", flush=True)
set_status("RUNNING", "Stage-07 initialization and JSON regression test complete")


JSON serialization self-test: PASS


STAGE 07 STATUS: RUNNING — Stage-07 initialization and JSON regression test complete


In [2]:
# ---------- Pure computational equivalence self-tests ----------
def _selected_multiplicity(order, row_counts, take):
    """Vectorized exact multiplicity of the fixed global top-k under row weights."""
    w = np.asarray(row_counts, dtype=np.int64)
    order = np.asarray(order, dtype=np.int64)
    take = int(take)
    total = int(w.sum())
    if take < 0 or take > total:
        raise ValueError("take outside weighted sample size")
    out = np.zeros_like(w)
    if take == 0:
        return out
    rw = w[order]
    cw = np.cumsum(rw)
    j = int(np.searchsorted(cw, take, side="left"))
    if j > 0:
        out[order[:j]] = rw[:j]
    prev = int(cw[j-1]) if j else 0
    out[order[j]] = take - prev
    if int(out.sum()) != take:
        raise RuntimeError("weighted selection failed to allocate requested top-k")
    return out

def _materialized_selected_multiplicity(order, row_counts, take):
    expanded = []
    for idx in np.asarray(order, int):
        expanded.extend([int(idx)] * int(row_counts[idx]))
    chosen = expanded[:int(take)]
    return np.bincount(chosen, minlength=len(row_counts)).astype(np.int64)

_rng_test = np.random.default_rng(1707)
for _ in range(100):
    n = int(_rng_test.integers(3, 20))
    w = _rng_test.integers(0, 5, size=n, dtype=np.int64)
    if w.sum() == 0:
        w[0] = 1
    order = _rng_test.permutation(n)
    take = int(_rng_test.integers(1, int(w.sum()) + 1))
    a = _selected_multiplicity(order, w, take)
    b = _materialized_selected_multiplicity(order, w, take)
    assert np.array_equal(a, b)

def _raw_slope(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    xc = x - x.mean(); den = float(xc @ xc)
    return float((xc @ (y-y.mean())) / den) if den > 0 else np.nan

def _platform_fe_slope(x, y, platform_codes):
    x = np.asarray(x, float); y = np.asarray(y, float); g = np.asarray(platform_codes, int)
    xr = np.empty_like(x); yr = np.empty_like(y)
    for k in np.unique(g):
        m = g == k
        xr[m] = x[m] - x[m].mean()
        yr[m] = y[m] - y[m].mean()
    den = float(xr @ xr)
    return float((xr @ yr) / den) if den > 0 else np.nan

def _spearman(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    rx = rankdata(x, method="average"); ry = rankdata(y, method="average")
    return _raw_slope((rx-rx.mean())/rx.std(ddof=0), (ry-ry.mean())/ry.std(ddof=0))

# FE residualization equivalence against an explicit dummy-variable least-squares fit.
for _ in range(50):
    x = _rng_test.normal(size=32); y = _rng_test.normal(size=32)
    g = np.tile(np.arange(4), 8)
    D = np.column_stack([np.ones(32), x, *(g == k for k in [1,2,3])])
    beta = np.linalg.lstsq(D, y, rcond=None)[0][1]
    assert np.isclose(_platform_fe_slope(x, y, g), beta, atol=1e-12, rtol=1e-10)

print("Weighted global-policy and platform-FE self-tests: PASS", flush=True)


Weighted global-policy and platform-FE self-tests: PASS


In [3]:
BRANCH_RULE=os.environ.get("SWARABYANJAN_BRANCH_RULE")
BRANCH_STAGE05_ROOT=Path(os.environ["SWARABYANJAN_BRANCH_STAGE05_ROOT"]).resolve() if os.environ.get("SWARABYANJAN_BRANCH_STAGE05_ROOT") else None
BRANCH_STAGE06_ROOT=Path(os.environ["SWARABYANJAN_BRANCH_STAGE06_ROOT"]).resolve() if os.environ.get("SWARABYANJAN_BRANCH_STAGE06_ROOT") else None
EXPECTED_STAGE05_CANONICAL_MANIFEST_SHA256="7246bc3a49a5f8d8472005cea9f6062d118431cd67d5dd2c4fd8f2ee31f52c27"
EXPECTED_STAGE06_CANONICAL_MANIFEST_SHA256="5a2962014445c723379b07fcf02e601644818c36e6c15fb7ee9478ca34711285"

# ---------- Exact canonical Stage-05 → Stage-06 provenance chain ----------
PRED_FILES = [
    "LOCKED_SOURCE_TEST_PREDICTIONS.parquet",
    "BANGLACLICK_PLATFORM_PREDICTIONS.parquet",
    "BAITBUSTER_HUMAN_PREDICTIONS.parquet",
]

EXPECTED_STAGE05 = {
    "stage_id": "05",
    "canonical_stage05": True,
    "canonical_version": "SWARABYANJAN-STAGE05-CANONICAL-vFINAL-1.0.1",
    "protocol_version": "SWARABYANJAN-CROSSPLATFORM-v1.0.0",
    "manifest_sha256": "7246bc3a49a5f8d8472005cea9f6062d118431cd67d5dd2c4fd8f2ee31f52c27",
    "checksum_sha256": "7171a0e581d8fb4bde2a630f4de631d74962954ee3890b8123d78a720823450a",
    "status_sha256": "005399d829bb66870810faa3f140757a7ee3d82eac4bae394fbbe79e701db8e6",
    "prediction_sha256": {
        "LOCKED_SOURCE_TEST_PREDICTIONS.parquet": "df863d7a24f7d451451822890de58613c3295eb0bcc1aa882150137b0aa9880a",
        "BANGLACLICK_PLATFORM_PREDICTIONS.parquet": "5685fd4142a0e8e7f85cd6040cf48042da2368d5f9596a7740c4f9f09c9519d5",
        "BAITBUSTER_HUMAN_PREDICTIONS.parquet": "a3e95820c62de00984a189e2c250f7fffd223ad4b3954db04a5368c7a544b7b9",
    },
}

EXPECTED_STAGE06 = {
    "stage_id": "06",
    "protocol_version": "SWARABYANJAN-CROSSPLATFORM-v1.0.0",
    "manifest_sha256": "5a2962014445c723379b07fcf02e601644818c36e6c15fb7ee9478ca34711285",
    "checksum_sha256": "5f8d0f7fc9cde5e0f159bf4c5f4ca8387441675407b4c43ab87a8063afac3beb",
    "status_sha256": "769f33b4771f3ec55700384ae6db7e9e2feb115d23717b27cbd9c17e4983538d",
    "h4_sha256": "299fc2e1394c80e1a090ac14d632e0f502b5b779c459e47dcb4e96130b621462",
    "upstream_gate_sha256": "0f98d1a2ac5cab5c692f6b35fd76af79dd72bf07f48ddfea26ab019cc89a12e7",
    "final_statistical_summary_sha256": "7da3348041074eb524fbdd7d9b8932779ffd69cf54d4b5a1af747b55acbe9f64",
    "bootstrap_runtime_sha256": "b9ff280b6eef366c1f74d63cd8aa4db0cecd23b9a6d07124f304a21e5a5b0785",
    "fixed_budget_tables_sha256": "3df30404ce1bcb8693b8cbabc0038bcc723aa2be3957c0c8b4ead9e93b19287b",
}

def _parse_sha256sums(path):
    out = {}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        parts = line.split(None, 1)
        if len(parts) != 2:
            hard_stop("Malformed checksum-contract line", {"file": str(path), "line": line})
        h, rel = parts
        rel = rel.lstrip("*").strip().replace("\\", "/")
        if not re.fullmatch(r"[0-9a-fA-F]{64}", h):
            hard_stop("Malformed checksum digest", {"file": str(path), "line": line})
        if rel in out:
            hard_stop("Duplicate checksum-contract path", {"file": str(path), "path": rel})
        out[rel] = h.lower()
    if not out:
        hard_stop("Empty checksum contract", {"file": str(path)})
    return out

def _verify_manifest_payload(manifest, label):
    if "manifest_payload_sha256" not in manifest:
        hard_stop(f"{label} manifest payload seal missing")
    recorded = manifest["manifest_payload_sha256"]
    if not isinstance(recorded, str) or not re.fullmatch(r"[0-9a-f]{64}", recorded):
        hard_stop(f"{label} manifest payload seal malformed")
    tmp = dict(manifest)
    tmp.pop("manifest_payload_sha256", None)
    observed = hashlib.sha256(
        json.dumps(_json_builtin(tmp), sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")
    ).hexdigest()
    if observed != recorded:
        hard_stop(f"{label} manifest payload seal mismatch", {"recorded": recorded, "observed": observed})
    return observed

def _verify_checksum_contract(root, sum_name, label):
    sums = _parse_sha256sums(root / sum_name)
    for rel, expected in sums.items():
        p = root / rel
        if not p.is_file():
            hard_stop(f"{label} checksum-contract artifact missing", {"artifact": rel})
        observed = sha256_file(p)
        if observed != expected:
            hard_stop(f"{label} checksum-contract artifact hash mismatch", {
                "artifact": rel, "expected": expected, "observed": observed
            })
    return sums

def _discover_exact_root(manifest_name, required_files, exact_manifest_sha256, label):
    matches = []
    seen = set()
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for p in root.rglob(manifest_name):
            d = p.parent.resolve()
            if d in seen:
                continue
            seen.add(d)
            if not all((d / f).is_file() for f in required_files):
                continue
            if sha256_file(d / manifest_name) == exact_manifest_sha256:
                matches.append(d)
    if not matches:
        hard_stop(f"Exact canonical {label} root not found", {
            "expected_manifest_sha256": exact_manifest_sha256,
            "searched_roots": [str(x) for x in SEARCH_ROOTS],
        })
    matches = sorted(set(matches), key=str)
    return matches[0], [str(x) for x in matches]

# ----- Stage 05 -----
if BRANCH_STAGE05_ROOT is not None:
    S5_ROOT=BRANCH_STAGE05_ROOT; S5_DUPLICATE_MOUNTS=[]
    S5_REQUIRED=["STAGE05_ARTIFACT_MANIFEST.json","STAGE05_SHA256SUMS.txt","STAGE_STATUS.json","H3_RESULTS.json",*PRED_FILES]
    if not all((S5_ROOT/f).is_file() for f in S5_REQUIRED): hard_stop("Branch Stage-05 root incomplete",{"root":str(S5_ROOT)})
    S5_MANIFEST=json.loads((S5_ROOT/"STAGE05_ARTIFACT_MANIFEST.json").read_text(encoding="utf-8"))
    _verify_manifest_payload(S5_MANIFEST,"Branch Stage-05")
    if S5_MANIFEST.get("canonical_stage05") is not False: hard_stop("Branch Stage-05 cannot be canonical")
    if S5_MANIFEST.get("base_canonical_stage05_manifest_sha256") != EXPECTED_STAGE05_CANONICAL_MANIFEST_SHA256: hard_stop("Branch Stage-05 lineage mismatch")
    S5_STATUS=json.loads((S5_ROOT/"STAGE_STATUS.json").read_text(encoding="utf-8"))
    if S5_STATUS.get("status")!="PASS": hard_stop("Branch Stage-05 status is not PASS")
    S5_SUMS=_verify_checksum_contract(S5_ROOT,"STAGE05_SHA256SUMS.txt","Branch Stage-05")
    S5_ARTIFACTS=S5_MANIFEST.get("artifacts",{})
    for f in ["LOCKED_SOURCE_TEST_PREDICTIONS.parquet","BAITBUSTER_HUMAN_PREDICTIONS.parquet"]:
        h=sha256_file(S5_ROOT/f)
        if S5_ARTIFACTS.get(f)!=h or S5_SUMS.get(f)!=h: hard_stop("Branch immutable Stage-05 prediction hash mismatch",{"artifact":f})
    PRED_ROOT=S5_ROOT; H3_ROOT=S5_ROOT
    PRED_IDENTITY={f:sha256_file(S5_ROOT/f) for f in PRED_FILES}
    H3_IDENTITY={"h3_results":sha256_file(S5_ROOT/"H3_RESULTS.json")}
else:

    S5_REQUIRED = [
        "STAGE05_ARTIFACT_MANIFEST.json", "STAGE05_SHA256SUMS.txt", "STAGE_STATUS.json",
        "H3_RESULTS.json", *PRED_FILES,
    ]
    S5_ROOT, S5_DUPLICATE_MOUNTS = _discover_exact_root(
        "STAGE05_ARTIFACT_MANIFEST.json", S5_REQUIRED, EXPECTED_STAGE05["manifest_sha256"], "Stage-05"
    )
    if sha256_file(S5_ROOT / "STAGE05_SHA256SUMS.txt") != EXPECTED_STAGE05["checksum_sha256"]:
        hard_stop("Stage-05 checksum-contract file identity mismatch")
    if sha256_file(S5_ROOT / "STAGE_STATUS.json") != EXPECTED_STAGE05["status_sha256"]:
        hard_stop("Stage-05 status-file identity mismatch")

    S5_MANIFEST = json.loads((S5_ROOT / "STAGE05_ARTIFACT_MANIFEST.json").read_text(encoding="utf-8"))
    _verify_manifest_payload(S5_MANIFEST, "Stage-05")
    for key in ["stage_id", "canonical_stage05", "canonical_version", "protocol_version", "artifacts"]:
        if key not in S5_MANIFEST:
            hard_stop("Stage-05 canonical identity/provenance field missing", {"field": key})
    if S5_MANIFEST["stage_id"] != EXPECTED_STAGE05["stage_id"]:
        hard_stop("Stage-05 stage_id mismatch")
    if S5_MANIFEST["canonical_stage05"] is not True:
        hard_stop("Stage-05 canonical_stage05 must be literal true")
    if S5_MANIFEST["canonical_version"] != EXPECTED_STAGE05["canonical_version"]:
        hard_stop("Stage-05 canonical version mismatch")
    if S5_MANIFEST["protocol_version"] != EXPECTED_STAGE05["protocol_version"]:
        hard_stop("Stage-05 protocol version mismatch")

    S5_STATUS = json.loads((S5_ROOT / "STAGE_STATUS.json").read_text(encoding="utf-8"))
    if "status" not in S5_STATUS or S5_STATUS["status"] != "PASS":
        hard_stop("Canonical Stage-05 status is not PASS")

    S5_SUMS = _verify_checksum_contract(S5_ROOT, "STAGE05_SHA256SUMS.txt", "Stage-05")
    S5_ARTIFACTS = S5_MANIFEST["artifacts"]
    if not isinstance(S5_ARTIFACTS, dict):
        hard_stop("Stage-05 manifest artifacts mapping malformed")

    for f, expected in EXPECTED_STAGE05["prediction_sha256"].items():
        observed = sha256_file(S5_ROOT / f)
        if observed != expected:
            hard_stop("Canonical Stage-05 prediction hash mismatch", {"file": f, "expected": expected, "observed": observed})
        if f not in S5_SUMS or S5_SUMS[f] != expected:
            hard_stop("Stage-05 checksum contract missing/wrong prediction hash", {"file": f})
        if f not in S5_ARTIFACTS or S5_ARTIFACTS[f] != expected:
            hard_stop("Stage-05 manifest missing/wrong prediction hash", {"file": f})

    for required_key in ["H3_RESULTS.json", "STAGE05_ARTIFACT_MANIFEST.json"]:
        if required_key not in S5_SUMS:
            hard_stop("Stage-05 checksum contract missing required artifact", {"artifact": required_key})

    PRED_ROOT = S5_ROOT
    H3_ROOT = S5_ROOT
    PRED_IDENTITY = dict(EXPECTED_STAGE05["prediction_sha256"])
    H3_IDENTITY = {"h3_results": sha256_file(S5_ROOT / "H3_RESULTS.json")}

# ----- Stage 06 -----
if BRANCH_STAGE06_ROOT is not None:
    S6_ROOT=BRANCH_STAGE06_ROOT; S6_DUPLICATE_MOUNTS=[]
    S6_REQUIRED=["REPRODUCIBILITY_MANIFEST.json","STAGE06_SHA256SUMS.txt","STAGE_STATUS.json","H4_RESULTS.json","UPSTREAM_PROVENANCE_GATE.json","FINAL_STATISTICAL_SUMMARY.json","BOOTSTRAP_RUNTIME.json","FIXED_BUDGET_TABLES.csv"]
    if not all((S6_ROOT/f).is_file() for f in S6_REQUIRED): hard_stop("Branch Stage-06 root incomplete",{"root":str(S6_ROOT)})
    S6_MANIFEST=json.loads((S6_ROOT/"REPRODUCIBILITY_MANIFEST.json").read_text(encoding="utf-8"))
    _verify_manifest_payload(S6_MANIFEST,"Branch Stage-06")
    S6_STATUS=json.loads((S6_ROOT/"STAGE_STATUS.json").read_text(encoding="utf-8"))
    if S6_STATUS.get("status")!="PASS": hard_stop("Branch Stage-06 status is not PASS")
    S6_UP=S6_MANIFEST.get("upstream",{})
    if S6_UP.get("mode")!="BRANCH_STAGE05_OUTPUT": hard_stop("Branch Stage-06 upstream mode mismatch")
    if S6_UP.get("branch_rule")!=BRANCH_RULE: hard_stop("Branch Stage-06 rule mismatch")
    if S6_UP.get("base_canonical_stage05_manifest_sha256")!=EXPECTED_STAGE05_CANONICAL_MANIFEST_SHA256: hard_stop("Branch Stage-06 base canonical Stage-05 lineage mismatch")
    if S6_UP.get("stage05_manifest_sha256")!=sha256_file(S5_ROOT/"STAGE05_ARTIFACT_MANIFEST.json"): hard_stop("Branch Stage-06 does not point to executed Stage-05 branch")
    if S6_UP.get("prediction_sha256")!=PRED_IDENTITY: hard_stop("Branch Stage-06 prediction identity mismatch")
    S6_SUMS=_verify_checksum_contract(S6_ROOT,"STAGE06_SHA256SUMS.txt","Branch Stage-06")
    S6_CONTRACT=S6_MANIFEST.get("scientific_contract",{})
    for key,expected in {"bootstrap_reps":5000,"budgets":[0.05,0.1,0.2],"primary_budget":0.1,"models":MODEL_KEYS,"seeds":SEEDS,"conditional_valid_only_bootstrap_inference":False,"rejection_resampling":False}.items():
        if S6_CONTRACT.get(key)!=expected: hard_stop("Branch Stage-06 scientific-contract mismatch",{"field":key})
    S6_GATE=json.loads((S6_ROOT/"UPSTREAM_PROVENANCE_GATE.json").read_text(encoding="utf-8"))
    if S6_GATE.get("status")!="PASS" or S6_GATE.get("mode")!="BRANCH_STAGE05_OUTPUT": hard_stop("Branch Stage-06 upstream gate mismatch")
    S6_IDENTITY={"manifest":sha256_file(S6_ROOT/"REPRODUCIBILITY_MANIFEST.json"),"checksum":sha256_file(S6_ROOT/"STAGE06_SHA256SUMS.txt"),"status":sha256_file(S6_ROOT/"STAGE_STATUS.json"),"h4_results":sha256_file(S6_ROOT/"H4_RESULTS.json")}
else:

    S6_REQUIRED = [
        "REPRODUCIBILITY_MANIFEST.json", "STAGE06_SHA256SUMS.txt", "STAGE_STATUS.json",
        "H4_RESULTS.json", "UPSTREAM_PROVENANCE_GATE.json", "FINAL_STATISTICAL_SUMMARY.json",
        "BOOTSTRAP_RUNTIME.json", "FIXED_BUDGET_TABLES.csv",
    ]
    S6_ROOT, S6_DUPLICATE_MOUNTS = _discover_exact_root(
        "REPRODUCIBILITY_MANIFEST.json", S6_REQUIRED, EXPECTED_STAGE06["manifest_sha256"], "Stage-06"
    )

    exact_s6_hashes = {
        "STAGE06_SHA256SUMS.txt": EXPECTED_STAGE06["checksum_sha256"],
        "STAGE_STATUS.json": EXPECTED_STAGE06["status_sha256"],
        "H4_RESULTS.json": EXPECTED_STAGE06["h4_sha256"],
        "UPSTREAM_PROVENANCE_GATE.json": EXPECTED_STAGE06["upstream_gate_sha256"],
        "FINAL_STATISTICAL_SUMMARY.json": EXPECTED_STAGE06["final_statistical_summary_sha256"],
        "BOOTSTRAP_RUNTIME.json": EXPECTED_STAGE06["bootstrap_runtime_sha256"],
        "FIXED_BUDGET_TABLES.csv": EXPECTED_STAGE06["fixed_budget_tables_sha256"],
    }
    for f, expected in exact_s6_hashes.items():
        observed = sha256_file(S6_ROOT / f)
        if observed != expected:
            hard_stop("Canonical Stage-06 artifact identity mismatch", {"file": f, "expected": expected, "observed": observed})

    S6_MANIFEST = json.loads((S6_ROOT / "REPRODUCIBILITY_MANIFEST.json").read_text(encoding="utf-8"))
    _verify_manifest_payload(S6_MANIFEST, "Stage-06")
    for key in ["stage_id", "protocol_version", "scientific_contract", "upstream"]:
        if key not in S6_MANIFEST:
            hard_stop("Stage-06 manifest required field missing", {"field": key})
    if S6_MANIFEST["stage_id"] != EXPECTED_STAGE06["stage_id"]:
        hard_stop("Stage-06 stage_id mismatch")
    if S6_MANIFEST["protocol_version"] != EXPECTED_STAGE06["protocol_version"]:
        hard_stop("Stage-06 protocol version mismatch")

    S6_STATUS = json.loads((S6_ROOT / "STAGE_STATUS.json").read_text(encoding="utf-8"))
    if "status" not in S6_STATUS or S6_STATUS["status"] != "PASS":
        hard_stop("Canonical Stage-06 status is not PASS")

    S6_SUMS = _verify_checksum_contract(S6_ROOT, "STAGE06_SHA256SUMS.txt", "Stage-06")
    for f in [
        "REPRODUCIBILITY_MANIFEST.json", "H4_RESULTS.json", "UPSTREAM_PROVENANCE_GATE.json",
        "FINAL_STATISTICAL_SUMMARY.json", "BOOTSTRAP_RUNTIME.json", "FIXED_BUDGET_TABLES.csv",
    ]:
        if f not in S6_SUMS:
            hard_stop("Stage-06 checksum contract missing required result artifact", {"artifact": f})
        if S6_SUMS[f] != sha256_file(S6_ROOT / f):
            hard_stop("Stage-06 checksum-contract hash disagreement", {"artifact": f})

    S6_CONTRACT = S6_MANIFEST["scientific_contract"]
    required_contract = {
        "bootstrap_reps": 5000,
        "budgets": [0.05, 0.1, 0.2],
        "primary_budget": 0.1,
        "models": MODEL_KEYS,
        "seeds": SEEDS,
        "conditional_valid_only_bootstrap_inference": False,
        "rejection_resampling": False,
    }
    for key, expected in required_contract.items():
        if key not in S6_CONTRACT:
            hard_stop("Stage-06 scientific-contract field missing", {"field": key})
        if S6_CONTRACT[key] != expected:
            hard_stop("Stage-06 scientific-contract mismatch", {"field": key, "expected": expected, "observed": S6_CONTRACT[key]})

    S6_UP = S6_MANIFEST["upstream"]
    for key in [
        "mode", "prediction_sha256", "stage05_manifest_sha256", "stage05_checksum_sha256",
        "stage05_status_sha256", "stage05_identity",
    ]:
        if key not in S6_UP:
            hard_stop("Stage-06 upstream provenance field missing", {"field": key})

    expected_s5_identity = {
        "stage_id": EXPECTED_STAGE05["stage_id"],
        "canonical_stage05": True,
        "canonical_version": EXPECTED_STAGE05["canonical_version"],
        "protocol_version": EXPECTED_STAGE05["protocol_version"],
    }
    if S6_UP["mode"] != "DIRECT_CANONICAL_STAGE05_OUTPUT":
        hard_stop("Stage-06 upstream mode is not canonical")
    if S6_UP["prediction_sha256"] != EXPECTED_STAGE05["prediction_sha256"]:
        hard_stop("Stage-06 prediction identity does not equal canonical Stage-05 prediction identity")
    if S6_UP["stage05_manifest_sha256"] != EXPECTED_STAGE05["manifest_sha256"]:
        hard_stop("Stage-06 Stage-05 manifest identity mismatch")
    if S6_UP["stage05_checksum_sha256"] != EXPECTED_STAGE05["checksum_sha256"]:
        hard_stop("Stage-06 Stage-05 checksum identity mismatch")
    if S6_UP["stage05_status_sha256"] != EXPECTED_STAGE05["status_sha256"]:
        hard_stop("Stage-06 Stage-05 status identity mismatch")
    if S6_UP["stage05_identity"] != expected_s5_identity:
        hard_stop("Stage-06 embedded canonical Stage-05 identity mismatch")

    S6_GATE = json.loads((S6_ROOT / "UPSTREAM_PROVENANCE_GATE.json").read_text(encoding="utf-8"))
    for key in [
        "status", "mode", "prediction_sha256", "stage05_manifest_sha256", "stage05_checksum_sha256",
        "stage05_status_sha256", "stage05_identity",
    ]:
        if key not in S6_GATE:
            hard_stop("Stage-06 upstream gate field missing", {"field": key})
    if S6_GATE["status"] != "PASS" or S6_GATE["mode"] != "DIRECT_CANONICAL_STAGE05_OUTPUT":
        hard_stop("Stage-06 upstream gate is not PASS/canonical")
    if S6_GATE["prediction_sha256"] != EXPECTED_STAGE05["prediction_sha256"]:
        hard_stop("Stage-06 upstream gate prediction identity mismatch")
    if S6_GATE["stage05_manifest_sha256"] != EXPECTED_STAGE05["manifest_sha256"]:
        hard_stop("Stage-06 upstream gate Stage-05 manifest mismatch")
    if S6_GATE["stage05_checksum_sha256"] != EXPECTED_STAGE05["checksum_sha256"]:
        hard_stop("Stage-06 upstream gate Stage-05 checksum mismatch")
    if S6_GATE["stage05_status_sha256"] != EXPECTED_STAGE05["status_sha256"]:
        hard_stop("Stage-06 upstream gate Stage-05 status mismatch")
    if S6_GATE["stage05_identity"] != expected_s5_identity:
        hard_stop("Stage-06 upstream gate Stage-05 identity mismatch")

    S6_IDENTITY = {
        "manifest": EXPECTED_STAGE06["manifest_sha256"],
        "checksum": EXPECTED_STAGE06["checksum_sha256"],
        "status": EXPECTED_STAGE06["status_sha256"],
        "h4_results": EXPECTED_STAGE06["h4_sha256"],
    }

if BRANCH_RULE:
    output_stage05_identity = {
        "stage_id":"05", "canonical_stage05":False,
        "canonical_version":EXPECTED_STAGE05["canonical_version"],
        "protocol_version":EXPECTED_STAGE05["protocol_version"],
        "manifest_sha256":sha256_file(S5_ROOT/"STAGE05_ARTIFACT_MANIFEST.json"),
        "checksum_sha256":sha256_file(S5_ROOT/"STAGE05_SHA256SUMS.txt"),
        "status_sha256":sha256_file(S5_ROOT/"STAGE_STATUS.json"),
        "prediction_sha256":PRED_IDENTITY,
        "h3_results_sha256":H3_IDENTITY["h3_results"],
    }
    output_stage06_identity = {
        "stage_id":"06", "protocol_version":PROTOCOL_VERSION,
        "manifest_sha256":S6_IDENTITY["manifest"],
        "checksum_sha256":S6_IDENTITY["checksum"],
        "status_sha256":S6_IDENTITY["status"],
        "h4_results_sha256":S6_IDENTITY["h4_results"],
    }
    output_mode="BRANCH_STAGE05_TO_STAGE06_CHAIN"
else:
    output_stage05_identity = {**expected_s5_identity, "manifest_sha256":EXPECTED_STAGE05["manifest_sha256"], "checksum_sha256":EXPECTED_STAGE05["checksum_sha256"], "status_sha256":EXPECTED_STAGE05["status_sha256"], "prediction_sha256":PRED_IDENTITY, "h3_results_sha256":H3_IDENTITY["h3_results"]}
    output_stage06_identity = {"stage_id":EXPECTED_STAGE06["stage_id"], "protocol_version":EXPECTED_STAGE06["protocol_version"], "manifest_sha256":EXPECTED_STAGE06["manifest_sha256"], "checksum_sha256":EXPECTED_STAGE06["checksum_sha256"], "status_sha256":EXPECTED_STAGE06["status_sha256"], "h4_results_sha256":EXPECTED_STAGE06["h4_sha256"]}
    output_mode="EXACT_CANONICAL_STAGE05_TO_STAGE06_CHAIN"

UPSTREAM_GATE = {
    "schema_version": 2, "mode": output_mode,
    "stage05_root": str(S5_ROOT), "stage06_root": str(S6_ROOT),
    "stage05_duplicate_mounts": S5_DUPLICATE_MOUNTS, "stage06_duplicate_mounts": S6_DUPLICATE_MOUNTS,
    "stage05_identity": output_stage05_identity, "stage06_identity": output_stage06_identity,
    "legacy_split_root_mode_enabled": False, "missing_provenance_fields_tolerated": False, "status": "PASS",
}
write_json(OUT_ROOT / "UPSTREAM_PROVENANCE_GATE.json", UPSTREAM_GATE)
print("Exact canonical Stage-05 → Stage-06 provenance chain: PASS", flush=True)


Exact canonical Stage-05 → Stage-06 provenance chain: PASS


In [4]:
# ---------- Load frozen inputs and enforce schema invariants ----------
SRC=pd.read_parquet(PRED_ROOT/"LOCKED_SOURCE_TEST_PREDICTIONS.parquet")
BC=pd.read_parquet(PRED_ROOT/"BANGLACLICK_PLATFORM_PREDICTIONS.parquet")
BB=pd.read_parquet(PRED_ROOT/"BAITBUSTER_HUMAN_PREDICTIONS.parquet")
H3=json.loads((H3_ROOT/"H3_RESULTS.json").read_text(encoding="utf-8"))
H4=json.loads((S6_ROOT/"H4_RESULTS.json").read_text(encoding="utf-8"))

required_cols={"row_id","label","cluster_id","model","seed","p_calibrated",*CUE_FAMILIES}
for name,df in [("source",SRC),("banglaclick",BC),("baitbuster",BB)]:
    miss=sorted(required_cols-set(df.columns))
    if miss: hard_stop("Prediction schema missing bridge columns",{"domain":name,"missing":miss})
    if sorted(df["model"].astype(str).unique().tolist()) != sorted(MODEL_KEYS):
        hard_stop("Unexpected model panel",{"domain":name,"models":sorted(df['model'].astype(str).unique().tolist())})
    if sorted(df["seed"].astype(int).unique().tolist()) != sorted(SEEDS):
        hard_stop("Unexpected seed panel",{"domain":name,"seeds":sorted(df['seed'].astype(int).unique().tolist())})

if sorted(BC["platform"].astype(str).unique().tolist()) != sorted(PLATFORMS):
    hard_stop("BanglaClick platform universe differs from frozen four-platform universe")

# Every model/seed copy of a domain must carry the same row IDs, labels, clusters and cues.
def canonical_item(df,name):
    g=(df[(df.model==MODEL_KEYS[0])&(df.seed.astype(int)==SEEDS[0])]
       .sort_values("row_id",kind="stable").reset_index(drop=True))
    if g.empty: hard_stop("Empty canonical domain",{"domain":name})
    ref=g[["row_id","label","cluster_id",*CUE_FAMILIES]].copy()
    for m in MODEL_KEYS:
        for seed in SEEDS:
            x=(df[(df.model==m)&(df.seed.astype(int)==seed)]
               .sort_values("row_id",kind="stable").reset_index(drop=True))
            if len(x)!=len(ref) or x["row_id"].astype(str).tolist()!=ref["row_id"].astype(str).tolist():
                hard_stop("Model/seed row replication mismatch",{"domain":name,"model":m,"seed":seed})
            if not np.array_equal(x["label"].to_numpy(int),ref["label"].to_numpy(int)):
                hard_stop("Model/seed label mismatch",{"domain":name,"model":m,"seed":seed})
            if x["cluster_id"].astype(str).tolist()!=ref["cluster_id"].astype(str).tolist():
                hard_stop("Model/seed cluster mismatch",{"domain":name,"model":m,"seed":seed})
            for cue in CUE_FAMILIES:
                if not np.array_equal((x[cue].to_numpy()>0),(ref[cue].to_numpy()>0)):
                    hard_stop("Model/seed cue mismatch",{"domain":name,"model":m,"seed":seed,"cue":cue})
    return g

SRC_ITEM=canonical_item(SRC,"source_test")
BC_ITEM={p:canonical_item(BC[BC.platform.eq(p)],f"banglaclick:{p}") for p in PLATFORMS}
BB_ITEM=canonical_item(BB,"baitbuster:youtube_human")

INPUT_SUMMARY={
    "source_rows":len(SRC_ITEM),
    "banglaclick_rows":{p:len(BC_ITEM[p]) for p in PLATFORMS},
    "baitbuster_rows":len(BB_ITEM),
    "models":MODEL_KEYS,"seeds":SEEDS,"cues":CUE_FAMILIES,"platforms":PLATFORMS,
}
write_json(OUT_ROOT/"FROZEN_INPUT_SUMMARY.json",INPUT_SUMMARY)
print("Frozen bridge inputs loaded and row-replication invariants: PASS", flush=True)


Frozen bridge inputs loaded and row-replication invariants: PASS


In [5]:
# ---------- Precompute cluster, H3 and H4 plans; no DataFrame work inside bootstrap ----------
class ClusterPlan:
    def __init__(self, cluster_ids, name):
        a=np.asarray(cluster_ids).astype(str)
        u=np.array(sorted(set(a)))
        code={x:i for i,x in enumerate(u)}
        self.row_cluster=np.asarray([code[x] for x in a],dtype=np.int32)
        self.n_clusters=len(u); self.n_rows=len(a); self.name=name
    def draw(self,rng):
        sampled=rng.choice(self.n_clusters,size=self.n_clusters,replace=True)
        cm=np.bincount(sampled,minlength=self.n_clusters)
        return cm[self.row_cluster].astype(np.int64,copy=False)

class H3Plan:
    def __init__(self, pred, name):
        item=(pred[(pred.model=="ebm")&(pred.seed.astype(int)==SEEDS[0])]
              .sort_values("row_id",kind="stable").reset_index(drop=True))
        self.name=name
        self.ids=item.row_id.astype(str).to_numpy()
        self.y=item.label.to_numpy(int)
        self.cues=np.vstack([(item[c].to_numpy()>0) for c in CUE_FAMILIES])
        self.cluster=ClusterPlan(item.cluster_id.to_numpy(),name)
        losses=[]
        for seed in SEEDS:
            g=(pred[(pred.model=="ebm")&(pred.seed.astype(int)==seed)]
               .sort_values("row_id",kind="stable").reset_index(drop=True))
            if g.row_id.astype(str).tolist()!=self.ids.tolist(): hard_stop("H3 seed alignment failure",{"domain":name,"seed":seed})
            p=np.clip(g.p_calibrated.to_numpy(float),1e-7,1-1e-7)
            losses.append(-(self.y*np.log(p)+(1-self.y)*np.log(1-p)))
        self.loss=np.vstack(losses)
        self.ones=np.ones(len(self.y),dtype=np.int64)
    def gaps(self,w):
        w=np.asarray(w,float)
        rd=np.full(len(CUE_FAMILIES),np.nan); ng=np.full((len(SEEDS),len(CUE_FAMILIES)),np.nan)
        for j,mask in enumerate(self.cues):
            wp=w[mask]; wa=w[~mask]; dp=float(wp.sum()); da=float(wa.sum())
            if dp<=0 or da<=0: continue
            rd[j]=float(np.dot(self.y[mask],wp)/dp - np.dot(self.y[~mask],wa)/da)
            ng[:,j]=(self.loss[:,mask]@wp)/dp - (self.loss[:,~mask]@wa)/da
        return rd,ng

class H4Plan:
    def __init__(self, g, name):
        frame=g.sort_values("row_id",kind="stable").reset_index(drop=True)
        self.name=name; self.n=len(frame)
        self.row_id=frame.row_id.astype(str).to_numpy()
        self.y=frame.label.to_numpy(int)
        self.p=frame.p_calibrated.to_numpy(float)
        self.err=((self.p>=0.5).astype(int)!=self.y).astype(np.int8)
        self.cues=np.vstack([(frame[c].to_numpy()>0) for c in CUE_FAMILIES]).astype(np.int8)
        u=1-np.maximum(self.p,1-self.p)
        idx=np.arange(self.n)
        # Exact Stage-06 tie-breaking: score then row_id, stable.
        tmp=pd.DataFrame({"i":idx,"row_id":self.row_id,"p":self.p,"u":u})
        self.pos_order=tmp.sort_values(["p","row_id"],ascending=[False,True],kind="stable")["i"].to_numpy(int)
        self.unc_order=tmp.sort_values(["u","row_id"],ascending=[False,True],kind="stable")["i"].to_numpy(int)
        # Precompute all score-order-dependent values once.
        self.pos_y_order=self.y[self.pos_order].astype(float,copy=False)
        self.pos_cue_order=(self.cues[:,self.pos_order]*self.y[self.pos_order][None,:]).astype(float,copy=False)
        self.unc_err_order=self.err[self.unc_order].astype(float,copy=False)
        self.unc_cue_order=(self.cues[:,self.unc_order]*self.err[self.unc_order][None,:]).astype(float,copy=False)
        self.cluster=ClusterPlan(frame.cluster_id.to_numpy(),name)
        self.ones=np.ones(self.n,dtype=np.int64)
    def metrics(self,w):
        w=np.asarray(w,dtype=np.int64)
        n=int(w.sum())
        if n<=0: return None
        total_pos=float(np.dot(self.y,w)); total_err=float(np.dot(self.err,w))
        take=max(1,int(np.ceil(PRIMARY_BUDGET*n)))
        # Exact weighted top-k using fixed global order; vectorized, with no bootstrap-time sorting.
        def prefix_capture(order, value_order, cue_value_order):
            rw=w[order]
            cw=np.cumsum(rw)
            j=int(np.searchsorted(cw,take,side="left"))
            prev=int(cw[j-1]) if j else 0
            partial=int(take-prev)
            total=float(np.dot(value_order[:j],rw[:j])) if j else 0.0
            if partial:
                total += float(value_order[j])*partial
            if j:
                by_cue=cue_value_order[:,:j] @ rw[:j]
            else:
                by_cue=np.zeros(len(CUE_FAMILIES),dtype=float)
            by_cue=np.asarray(by_cue,float)
            if partial:
                by_cue += cue_value_order[:,j]*partial
            return total,by_cue

        captured_pos,cue_pos_capture=prefix_capture(self.pos_order,self.pos_y_order,self.pos_cue_order)
        captured_err,cue_err_capture=prefix_capture(self.unc_order,self.unc_err_order,self.unc_cue_order)
        prevalence=total_pos/n
        global_lift=(captured_pos/take)/prevalence if total_pos>0 else np.nan
        global_error=captured_err/total_err if total_err>0 else np.nan
        cue_lift=np.full(len(CUE_FAMILIES),np.nan)
        cue_error=np.full(len(CUE_FAMILIES),np.nan)
        if total_pos>0:
            cue_lift=(cue_pos_capture/take)/prevalence
        if total_err>0:
            cue_error=cue_err_capture/total_err
        return {"global_lift":global_lift,"global_error_capture":global_error,
                "cue_lift":cue_lift,"cue_error_capture":cue_error,
                "n":n,"take":take,"total_pos":total_pos,"total_err":total_err}

H3_SRC=H3Plan(SRC,"source_test")
H3_BC={p:H3Plan(BC[BC.platform.eq(p)],f"banglaclick:{p}") for p in PLATFORMS}

H4_PLANS={}
for dname,df in {"source_test":SRC, **{f"banglaclick:{p}":BC[BC.platform.eq(p)] for p in PLATFORMS}, "baitbuster:youtube_human":BB}.items():
    for m in MODEL_KEYS:
        for seed in SEEDS:
            g=df[(df.model==m)&(df.seed.astype(int)==seed)]
            H4_PLANS[(dname,m,seed)]=H4Plan(g,f"{dname}|{m}|{seed}")

# H3 and H4 canonical arrays must also be row/cluster aligned because the joint draw is shared.
for dname,h3p in [("source_test",H3_SRC), *[(f"banglaclick:{p}",H3_BC[p]) for p in PLATFORMS]]:
    h4p=H4_PLANS[(dname,MODEL_KEYS[0],SEEDS[0])]
    if h3p.ids.tolist()!=h4p.row_id.tolist() or not np.array_equal(h3p.cluster.row_cluster,h4p.cluster.row_cluster):
        hard_stop("H3/H4 canonical row or cluster alignment failed",{"domain":dname})

# Domain cluster structures must be identical across models/seeds, allowing one joint draw/domain/replicate.
for dname in ["source_test",*[f"banglaclick:{p}" for p in PLATFORMS],"baitbuster:youtube_human"]:
    ref=H4_PLANS[(dname,MODEL_KEYS[0],SEEDS[0])].cluster.row_cluster
    for m in MODEL_KEYS:
        for seed in SEEDS:
            if not np.array_equal(ref,H4_PLANS[(dname,m,seed)].cluster.row_cluster):
                hard_stop("Cluster coding differs across model/seed copies",{"domain":dname,"model":m,"seed":seed})
print(f"Precomputed H4 plans: {len(H4_PLANS)}; no bootstrap-time sorting/groupby/DataFrame construction", flush=True)


Precomputed H4 plans: 120; no bootstrap-time sorting/groupby/DataFrame construction


In [6]:
# ---------- H3 point-estimate parity gate: all 32 frozen cells ----------
def _close(a,b):
    if a is None or b is None: return a is None and b is None
    return bool(np.isclose(float(a),float(b),atol=PARITY_ATOL,rtol=PARITY_RTOL))

h3_lookup={(r["cue"],str(r["target"]).split(":",1)[1]):r for r in H3["banglaclick_cue_platform_contrasts"]}
srd,snll=H3_SRC.gaps(H3_SRC.ones)
parity_rows=[]
for p in PLATFORMS:
    trd,tnll=H3_BC[p].gaps(H3_BC[p].ones)
    for j,cue in enumerate(CUE_FAMILIES):
        obs_rd=float(trd[j]-srd[j]) if np.isfinite(trd[j]) and np.isfinite(srd[j]) else None
        by_seed=tnll[:,j]-snll[:,j]
        obs_nll=float(np.mean(by_seed)) if np.isfinite(by_seed).all() else None
        frozen=h3_lookup[(cue,p)]
        is_branch_news=bool(BRANCH_RULE) and p=="news"
        ok_rd=_close(obs_rd,frozen.get("delta_rd")); ok_nll=_close(obs_nll,frozen.get("ebm_delta_nll_gap_mean_5seeds"))
        parity_rows.append({"cue":cue,"platform":p,"computed_delta_rd":obs_rd,"frozen_delta_rd":frozen.get("delta_rd"),
                            "delta_rd_match":ok_rd,"computed_delta_nll_gap":obs_nll,"frozen_delta_nll_gap":frozen.get("ebm_delta_nll_gap_mean_5seeds"),
                            "delta_nll_gap_match":ok_nll,"match":(None if is_branch_news else bool(ok_rd and ok_nll)),"status":("BRANCH_EXPECTED_DIFFERENCE" if is_branch_news else "EXACT_NUMERICAL_PARITY")})
H3_PARITY={"schema_version":1,"cells_expected":32,"cells_checked":len(parity_rows),"cells_matching":sum(bool(r["match"]) for r in parity_rows if r["match"] is not None),
           "all_match":bool(all(r["match"] is not False for r in parity_rows)),"atol":PARITY_ATOL,"rtol":PARITY_RTOL,"rows":parity_rows}
write_json(OUT_ROOT/"H3_POINT_PARITY.json",H3_PARITY)
if not H3_PARITY["all_match"]: hard_stop("H3 point-estimate parity failed",{"matching":H3_PARITY["cells_matching"]})
print("H3 point-estimate parity: PASS (32/32 cells)",flush=True)


H3 point-estimate parity: PASS (32/32 cells)


In [7]:
# ---------- Stage-06 global 10% policy point-parity gate: 100 source-target settings ----------
h4_rows=[r for r in H4.get("bootstrap_rows",[]) if bool(r.get("primary_budget")) and np.isclose(float(r.get("budget")),PRIMARY_BUDGET)]
h4_lookup={(str(r["domain"]),str(r["model"]),int(r["seed"])):r for r in h4_rows}
parity=[]
target_domains=[f"banglaclick:{p}" for p in PLATFORMS]+["baitbuster:youtube_human"]
for dname in target_domains:
    for m in MODEL_KEYS:
        for seed in SEEDS:
            s=H4_PLANS[("source_test",m,seed)].metrics(H4_PLANS[("source_test",m,seed)].ones)
            t=H4_PLANS[(dname,m,seed)].metrics(H4_PLANS[(dname,m,seed)].ones)
            fr=h4_lookup.get((dname,m,seed))
            if fr is None: hard_stop("Stage-06 primary 10% row missing",{"domain":dname,"model":m,"seed":seed})
            is_branch_news=bool(BRANCH_RULE) and dname=="banglaclick:news"
            checks={
                "source_lift":_close(s["global_lift"],fr.get("source_lift")),
                "target_lift":_close(t["global_lift"],fr.get("target_lift")),
                "source_error_capture":_close(s["global_error_capture"],fr.get("source_error_capture")),
                "target_error_capture":_close(t["global_error_capture"],fr.get("target_error_capture")),
            }
            parity.append({"domain":dname,"model":m,"seed":seed,**checks,"match":(None if is_branch_news else bool(all(checks.values()))),"status":("BRANCH_EXPECTED_DIFFERENCE" if is_branch_news else "EXACT_NUMERICAL_PARITY")})
H4_PARITY={"schema_version":2,"policy":"global 10% unchanged; no cue-specific reranking","settings_expected":100,"settings_checked":len(parity),
           "settings_matching":sum(bool(r["match"]) for r in parity if r["match"] is not None),"all_match":bool(all(r["match"] is not False for r in parity)),
           "atol":PARITY_ATOL,"rtol":PARITY_RTOL,"rows":parity}
# This write path previously failed on a NumPy boolean. write_json recursively converts np.bool_ and was regression-tested above.
write_json(OUT_ROOT/"H4_POLICY_POINT_PARITY.json",H4_PARITY)
# Reload to prove persistence, not merely in-memory construction.
_h4_parity_reload=json.loads((OUT_ROOT/"H4_POLICY_POINT_PARITY.json").read_text(encoding="utf-8"))
if not _h4_parity_reload["all_match"]: hard_stop("H4 global 10% policy point parity failed",{"matching":H4_PARITY["settings_matching"]})
print("H4 global 10% point parity: PASS (100/100 settings)",flush=True)


H4 global 10% point parity: PASS (100/100 settings)


In [8]:
# ---------- Bridge point estimates across the fixed 32-cell universe ----------
def panel_cue_metrics(domain, row_counts):
    """Average 5 seeds within model, then equally average the 4 locked model families."""
    per_model_lift=[]; per_model_error=[]
    for m in MODEL_KEYS:
        seed_lift=[]; seed_error=[]
        for seed in SEEDS:
            z=H4_PLANS[(domain,m,seed)].metrics(row_counts)
            if z is None:
                return None,None
            seed_lift.append(z["cue_lift"]); seed_error.append(z["cue_error_capture"])
        per_model_lift.append(np.mean(np.vstack(seed_lift),axis=0))
        per_model_error.append(np.mean(np.vstack(seed_error),axis=0))
    return np.mean(np.vstack(per_model_lift),axis=0),np.mean(np.vstack(per_model_error),axis=0)

src_w=H4_PLANS[("source_test",MODEL_KEYS[0],SEEDS[0])].ones
src_lift,src_err=panel_cue_metrics("source_test",src_w)

cell_rows=[]
for pi,p in enumerate(PLATFORMS):
    tdom=f"banglaclick:{p}"
    tw=H4_PLANS[(tdom,MODEL_KEYS[0],SEEDS[0])].ones
    tgt_lift,tgt_err=panel_cue_metrics(tdom,tw)
    for j,cue in enumerate(CUE_FAMILIES):
        h3r=h3_lookup[(cue,p)]
        cell_rows.append({
            "cue":cue,"platform":p,
            "abs_delta_rd":abs(float(h3r["delta_rd"])),
            "abs_delta_nll_gap":abs(float(h3r["ebm_delta_nll_gap_mean_5seeds"])),
            "source_cue_lift":float(src_lift[j]),"target_cue_lift":float(tgt_lift[j]),
            "lift_deterioration_source_minus_target":float(src_lift[j]-tgt_lift[j]),
            "source_cue_error_capture":float(src_err[j]),"target_cue_error_capture":float(tgt_err[j]),
            "error_capture_deterioration_source_minus_target":float(src_err[j]-tgt_err[j]),
        })
CELL=pd.DataFrame(cell_rows)
if len(CELL)!=32 or CELL[["cue","platform"]].duplicated().any(): hard_stop("Bridge cell universe is not exactly 32 unique cells")
CELL.to_csv(OUT_ROOT/"BRIDGE_32_CELL_POINT_TABLE.csv",index=False)

PLATFORM_CODES=np.asarray([PLATFORMS.index(p) for p in CELL.platform],dtype=int)
PAIR_DEFS=[
    ("abs_delta_rd__vs__lift_deterioration","abs_delta_rd","lift_deterioration_source_minus_target"),
    ("abs_delta_rd__vs__error_capture_deterioration","abs_delta_rd","error_capture_deterioration_source_minus_target"),
    ("abs_delta_nll_gap__vs__lift_deterioration","abs_delta_nll_gap","lift_deterioration_source_minus_target"),
    ("abs_delta_nll_gap__vs__error_capture_deterioration","abs_delta_nll_gap","error_capture_deterioration_source_minus_target"),
]

def association_stats(x,y):
    x=np.asarray(x,float);y=np.asarray(y,float)
    if len(x)!=32 or not np.isfinite(x).all() or not np.isfinite(y).all():
        return {"spearman_rho":np.nan,"raw_ols_slope":np.nan,"platform_fe_slope":np.nan}
    return {"spearman_rho":_spearman(x,y),"raw_ols_slope":_raw_slope(x,y),"platform_fe_slope":_platform_fe_slope(x,y,PLATFORM_CODES)}

POINT_ASSOC={name:association_stats(CELL[x].to_numpy(),CELL[y].to_numpy()) for name,x,y in PAIR_DEFS}
write_json(OUT_ROOT/"BRIDGE_POINT_ASSOCIATIONS.json",{"schema_version":1,"exploratory":True,"pairs":POINT_ASSOC})
print("Bridge point estimates computed for fixed 32-cell universe",flush=True)


Bridge point estimates computed for fixed 32-cell universe


In [9]:
# ---------- Joint 5,000-replicate cluster bootstrap ----------
# One draw per domain per replicate; identical row multiplicities are reused across every model/seed/cue endpoint.
rng=np.random.default_rng(BOOTSTRAP_SEED)
boot={name:{stat:np.full(BOOTSTRAP_REPS,np.nan) for stat in ["spearman_rho","raw_ols_slope","platform_fe_slope"]} for name,_,_ in PAIR_DEFS}
reasons={name:Counter() for name,_,_ in PAIR_DEFS}
start=time.monotonic()

for rep in range(BOOTSTRAP_REPS):
    # Joint domain resamples.
    sw=H3_SRC.cluster.draw(rng)
    tw={p:H3_BC[p].cluster.draw(rng) for p in PLATFORMS}

    # H3 x-coordinates under the same source/platform draws.
    srd,sng=H3_SRC.gaps(sw)
    xrd=[];xnll=[]
    for p in PLATFORMS:
        trd,tng=H3_BC[p].gaps(tw[p])
        for j in range(len(CUE_FAMILIES)):
            xrd.append(abs(trd[j]-srd[j]) if np.isfinite(trd[j]) and np.isfinite(srd[j]) else np.nan)
            d=tng[:,j]-sng[:,j]
            xnll.append(abs(float(np.mean(d))) if np.isfinite(d).all() else np.nan)

    # H3 group support is shared by ΔRD and ΔNLL-gap. If any prespecified H3 cell is
    # undefined, all four 32-cell association replicates are invalid, so H4 work can be skipped.
    if not np.isfinite(np.asarray(xrd,float)).all() or not np.isfinite(np.asarray(xnll,float)).all():
        for name,_,_ in PAIR_DEFS:
            reasons[name]["h3_nonfinite_in_prespecified_32_cells"] += 1
        if (rep+1)%500==0 or rep+1==BOOTSTRAP_REPS:
            elapsed=time.monotonic()-start
            print(f"[bridge bootstrap] {rep+1}/{BOOTSTRAP_REPS} elapsed={elapsed:.1f}s",flush=True)
        continue

    # H4 y-coordinates. H4 row order is aligned to the H3 canonical item order by construction.
    src_lift_b,src_err_b=panel_cue_metrics("source_test",sw)
    ylift=[];yerr=[]
    for p in PLATFORMS:
        tl,te=panel_cue_metrics(f"banglaclick:{p}",tw[p])
        if src_lift_b is None or tl is None:
            ylift.extend([np.nan]*len(CUE_FAMILIES))
        else:
            ylift.extend((src_lift_b-tl).tolist())
        if src_err_b is None or te is None:
            yerr.extend([np.nan]*len(CUE_FAMILIES))
        else:
            yerr.extend((src_err_b-te).tolist())

    # Reorder cue/platform vectors to match CELL (which is platform-major then cue-major).
    # H3 vectors above are platform-major/cue-major as well.
    xmap={"abs_delta_rd":np.asarray(xrd,float),"abs_delta_nll_gap":np.asarray(xnll,float)}
    ymap={"lift_deterioration_source_minus_target":np.asarray(ylift,float),"error_capture_deterioration_source_minus_target":np.asarray(yerr,float)}

    for name,xname,yname in PAIR_DEFS:
        x=xmap[xname];y=ymap[yname]
        if len(x)!=32 or len(y)!=32:
            reasons[name]["wrong_cell_count"]+=1; continue
        if not np.isfinite(x).all():
            reasons[name]["h3_nonfinite_in_prespecified_32_cells"]+=1; continue
        if not np.isfinite(y).all():
            reasons[name]["h4_nonfinite_in_prespecified_32_cells"]+=1; continue
        st=association_stats(x,y)
        if not all(np.isfinite(v) for v in st.values()):
            reasons[name]["association_statistic_nonfinite"]+=1; continue
        for stat,v in st.items(): boot[name][stat][rep]=v

    if (rep+1)%500==0 or rep+1==BOOTSTRAP_REPS:
        elapsed=time.monotonic()-start
        print(f"[bridge bootstrap] {rep+1}/{BOOTSTRAP_REPS} elapsed={elapsed:.1f}s",flush=True)

BOOTSTRAP_SECONDS=float(time.monotonic()-start)

def finalize(arr,reason_counter):
    a=np.asarray(arr,float)
    if len(a)!=BOOTSTRAP_REPS: raise RuntimeError("bootstrap attempt count mismatch")
    valid=int(np.isfinite(a).sum()); invalid=BOOTSTRAP_REPS-valid
    # One coarse reason is recorded per invalid pair-replicate before any statistic is written.
    if int(sum(reason_counter.values()))!=invalid:
        raise RuntimeError(f"invalid-reason accounting mismatch: reasons={sum(reason_counter.values())}, invalid={invalid}")
    estimable=invalid==0
    return {
        "bootstrap_attempts":BOOTSTRAP_REPS,"valid_replicates":valid,"invalid_replicates":invalid,"invalid_rate":invalid/BOOTSTRAP_REPS,
        "invalid_reason_counts":dict(sorted(reason_counter.items())),
        "estimability_status":"ESTIMABLE" if estimable else "NON_ESTIMABLE_UNDER_LOCKED_JOINT_BOOTSTRAP",
        "ci95":[float(np.percentile(a,2.5)),float(np.percentile(a,97.5))] if estimable else None,
        "p_two_sided":float(min(1.0,2*min(np.mean(a<=0),np.mean(a>=0)))) if estimable else None,
    }

BRIDGE_RESULTS={"schema_version":2,"bridge_version":BRIDGE_VERSION,"exploratory_secondary":True,"causal":False,
    "cells":32,"bootstrap_replicates":BOOTSTRAP_REPS,"bootstrap_seed":BOOTSTRAP_SEED,
    "global_review_policy":"10% global; no cue-specific reranking or quota",
    "seed_handling":"five seeds averaged within model; four locked model families equally weighted",
    "nonestimability_policy":"any invalid prespecified component in a replicate invalidates that association replicate; no NaN filtering, rejection resampling, or cell deletion",
    "pairs":{},"runtime_seconds":BOOTSTRAP_SECONDS}
for name,_,_ in PAIR_DEFS:
    BRIDGE_RESULTS["pairs"][name]={"point":POINT_ASSOC[name],"bootstrap":{}}
    for stat in ["spearman_rho","raw_ols_slope","platform_fe_slope"]:
        BRIDGE_RESULTS["pairs"][name]["bootstrap"][stat]=finalize(boot[name][stat],reasons[name])
write_json(OUT_ROOT/"H3_H4_BRIDGE_RESULTS.json",BRIDGE_RESULTS)
write_json(OUT_ROOT/"BOOTSTRAP_RUNTIME.json",{"reps":BOOTSTRAP_REPS,"elapsed_seconds":BOOTSTRAP_SECONDS})
print(f"Joint bridge bootstrap complete in {BOOTSTRAP_SECONDS:.1f}s",flush=True)


[bridge bootstrap] 500/5000 elapsed=2.1s


[bridge bootstrap] 1000/5000 elapsed=4.0s


[bridge bootstrap] 1500/5000 elapsed=6.2s


[bridge bootstrap] 2000/5000 elapsed=8.3s


[bridge bootstrap] 2500/5000 elapsed=10.4s


...

[bridge bootstrap] 3500/5000 elapsed=14.6s


[bridge bootstrap] 4000/5000 elapsed=16.6s


[bridge bootstrap] 4500/5000 elapsed=18.6s


[bridge bootstrap] 5000/5000 elapsed=20.7s


Joint bridge bootstrap complete in 20.7s


In [10]:
# ---------- Secondary robustness only: leave-one-platform-out platform-FE slopes ----------
# This cell does NOT alter the primary 32-cell associations or their bootstrap.
# A fresh RNG with the same locked seed and the same domain-draw order reproduces
# the identical joint source/platform resamples used by the primary bootstrap.
LOO_STATS = {}
LOO_POINT_ROWS = []

for name, xname, yname in PAIR_DEFS:
    x_all = CELL[xname].to_numpy(float)
    y_all = CELL[yname].to_numpy(float)
    LOO_STATS[name] = {}
    for omitted in PLATFORMS:
        keep = (CELL["platform"].to_numpy(str) != omitted)
        if int(keep.sum()) != 24:
            hard_stop("LOO retained-cell count is not 24", {"pair": name, "omitted_platform": omitted})
        slope = _platform_fe_slope(x_all[keep], y_all[keep], PLATFORM_CODES[keep])
        if not np.isfinite(slope):
            hard_stop("LOO point platform-FE slope is non-finite", {"pair": name, "omitted_platform": omitted})
        LOO_STATS[name][omitted] = {
            "point_platform_fe_slope": float(slope),
            "bootstrap_values": np.full(BOOTSTRAP_REPS, np.nan),
            "invalid_reasons": Counter(),
        }
        LOO_POINT_ROWS.append({
            "pair": name,
            "omitted_platform": omitted,
            "retained_platforms": "|".join([p for p in PLATFORMS if p != omitted]),
            "retained_cells": 24,
            "point_platform_fe_slope": float(slope),
            "secondary_robustness_only": True,
        })

pd.DataFrame(LOO_POINT_ROWS).to_csv(OUT_ROOT / "LEAVE_ONE_PLATFORM_OUT_POINT_TABLE.csv", index=False)

rng_loo = np.random.default_rng(BOOTSTRAP_SEED)
loo_start = time.monotonic()

for rep in range(BOOTSTRAP_REPS):
    # EXACT same joint resampling law and draw order as the primary bootstrap.
    sw = H3_SRC.cluster.draw(rng_loo)
    tw = {p: H3_BC[p].cluster.draw(rng_loo) for p in PLATFORMS}

    srd, sng = H3_SRC.gaps(sw)
    xrd = []
    xnll = []
    for p in PLATFORMS:
        trd, tng = H3_BC[p].gaps(tw[p])
        for j in range(len(CUE_FAMILIES)):
            xrd.append(abs(trd[j] - srd[j]) if np.isfinite(trd[j]) and np.isfinite(srd[j]) else np.nan)
            d = tng[:, j] - sng[:, j]
            xnll.append(abs(float(np.mean(d))) if np.isfinite(d).all() else np.nan)

    src_lift_b, src_err_b = panel_cue_metrics("source_test", sw)
    ylift = []
    yerr = []
    for p in PLATFORMS:
        tl, te = panel_cue_metrics(f"banglaclick:{p}", tw[p])
        if src_lift_b is None or tl is None:
            ylift.extend([np.nan] * len(CUE_FAMILIES))
        else:
            ylift.extend((src_lift_b - tl).tolist())
        if src_err_b is None or te is None:
            yerr.extend([np.nan] * len(CUE_FAMILIES))
        else:
            yerr.extend((src_err_b - te).tolist())

    xmap = {
        "abs_delta_rd": np.asarray(xrd, float),
        "abs_delta_nll_gap": np.asarray(xnll, float),
    }
    ymap = {
        "lift_deterioration_source_minus_target": np.asarray(ylift, float),
        "error_capture_deterioration_source_minus_target": np.asarray(yerr, float),
    }

    for name, xname, yname in PAIR_DEFS:
        x = xmap[xname]
        y = ymap[yname]
        if len(x) != 32 or len(y) != 32:
            for omitted in PLATFORMS:
                LOO_STATS[name][omitted]["invalid_reasons"]["wrong_primary_cell_count"] += 1
            continue

        for omitted in PLATFORMS:
            keep = (CELL["platform"].to_numpy(str) != omitted)
            xs = x[keep]
            ys = y[keep]
            codes = PLATFORM_CODES[keep]

            if len(xs) != 24 or len(ys) != 24:
                LOO_STATS[name][omitted]["invalid_reasons"]["wrong_retained_cell_count"] += 1
                continue
            if not np.isfinite(xs).all():
                LOO_STATS[name][omitted]["invalid_reasons"]["h3_nonfinite_in_retained_24_cells"] += 1
                continue
            if not np.isfinite(ys).all():
                LOO_STATS[name][omitted]["invalid_reasons"]["h4_nonfinite_in_retained_24_cells"] += 1
                continue

            slope = _platform_fe_slope(xs, ys, codes)
            if not np.isfinite(slope):
                LOO_STATS[name][omitted]["invalid_reasons"]["platform_fe_slope_nonfinite"] += 1
                continue
            LOO_STATS[name][omitted]["bootstrap_values"][rep] = float(slope)

    if (rep + 1) % 500 == 0 or rep + 1 == BOOTSTRAP_REPS:
        print(f"[LOO secondary bootstrap] {rep+1}/{BOOTSTRAP_REPS} elapsed={time.monotonic()-loo_start:.1f}s", flush=True)

LOO_SECONDS = float(time.monotonic() - loo_start)

def _finalize_loo(arr, reason_counter):
    a = np.asarray(arr, float)
    if len(a) != BOOTSTRAP_REPS:
        raise RuntimeError("LOO bootstrap attempt count mismatch")
    valid = int(np.isfinite(a).sum())
    invalid = BOOTSTRAP_REPS - valid
    if int(sum(reason_counter.values())) != invalid:
        raise RuntimeError(
            f"LOO invalid-reason accounting mismatch: reasons={sum(reason_counter.values())}, invalid={invalid}"
        )
    estimable = invalid == 0
    return {
        "bootstrap_attempts": BOOTSTRAP_REPS,
        "valid_replicates": valid,
        "invalid_replicates": invalid,
        "invalid_rate": invalid / BOOTSTRAP_REPS,
        "invalid_reason_counts": dict(sorted(reason_counter.items())),
        "estimability_status": "ESTIMABLE" if estimable else "NON_ESTIMABLE_UNDER_LOCKED_JOINT_BOOTSTRAP",
        "ci95": [float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))] if estimable else None,
        "p_two_sided": float(min(1.0, 2 * min(np.mean(a <= 0), np.mean(a >= 0)))) if estimable else None,
    }

LOO_RESULTS = {
    "schema_version": 1,
    "bridge_version": BRIDGE_VERSION,
    "analysis_role": "SECONDARY_ROBUSTNESS_ONLY",
    "exploratory_secondary": True,
    "causal": False,
    "primary_bridge_modified": False,
    "primary_cell_universe": 32,
    "retained_cells_per_loo_fit": 24,
    "omitted_platforms": PLATFORMS,
    "association_statistic": "platform_fe_slope",
    "pairs": {},
    "bootstrap_replicates": BOOTSTRAP_REPS,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "resampling_identity": "same seed and same source/platform draw order as primary joint bootstrap",
    "nonestimability_policy": "any invalid component in the retained 24-cell subset invalidates that LOO pair/omitted-platform replicate; no filtering or redraw",
    "runtime_seconds": LOO_SECONDS,
}

for name, _, _ in PAIR_DEFS:
    LOO_RESULTS["pairs"][name] = {}
    for omitted in PLATFORMS:
        z = LOO_STATS[name][omitted]
        LOO_RESULTS["pairs"][name][omitted] = {
            "point_platform_fe_slope": z["point_platform_fe_slope"],
            "bootstrap": _finalize_loo(z["bootstrap_values"], z["invalid_reasons"]),
        }

write_json(OUT_ROOT / "LEAVE_ONE_PLATFORM_OUT_ROBUSTNESS.json", LOO_RESULTS)
print(f"Secondary LOO robustness complete in {LOO_SECONDS:.1f}s", flush=True)


[LOO secondary bootstrap] 500/5000 elapsed=3.8s


[LOO secondary bootstrap] 1000/5000 elapsed=7.5s


[LOO secondary bootstrap] 1500/5000 elapsed=11.4s


[LOO secondary bootstrap] 2000/5000 elapsed=15.2s


[LOO secondary bootstrap] 2500/5000 elapsed=18.9s


...

[LOO secondary bootstrap] 3500/5000 elapsed=26.7s


[LOO secondary bootstrap] 4000/5000 elapsed=30.6s


[LOO secondary bootstrap] 4500/5000 elapsed=34.3s


[LOO secondary bootstrap] 5000/5000 elapsed=38.0s


Secondary LOO robustness complete in 38.0s


In [11]:
# ---------- Final package, checksums and PASS ----------
required=[
    "UPSTREAM_PROVENANCE_GATE.json","FROZEN_INPUT_SUMMARY.json","H3_POINT_PARITY.json","H4_POLICY_POINT_PARITY.json",
    "BRIDGE_32_CELL_POINT_TABLE.csv","BRIDGE_POINT_ASSOCIATIONS.json","H3_H4_BRIDGE_RESULTS.json","BOOTSTRAP_RUNTIME.json",
    "LEAVE_ONE_PLATFORM_OUT_POINT_TABLE.csv","LEAVE_ONE_PLATFORM_OUT_ROBUSTNESS.json",
]
missing=[x for x in required if not (OUT_ROOT/x).is_file()]
if missing: hard_stop("Stage-07 required artifact missing",{"missing":missing})

summary_lines=[
    "# Stage 07 H3→H4 exploratory bridge summary","",
    "- Status: completed as a secondary/exploratory observational analysis.",
    "- Fixed universe: 8 cue families × 4 BanglaClick platforms = 32 cells.",
    "- H3 point parity: 32/32 required.",
    "- Stage-06 global 10% policy parity: 100/100 settings required.",
    "- Review policy: global 10%; no cue-specific reranking.",
    "- Five seeds are averaged within model; four model families are equally weighted.",
    "- Joint cluster bootstrap: 5,000 attempts; invalid replicates are counted and never silently filtered.",
    "- Association is descriptive/exploratory and non-causal; it cannot rescue non-estimable confirmatory H3 inference.",
    "- Leave-one-platform-out platform-FE analysis is secondary robustness only; it does not alter the four primary association pairs or the fixed 32-cell primary universe.",
]
for name in BRIDGE_RESULTS["pairs"]:
    fe=BRIDGE_RESULTS["pairs"][name]["bootstrap"]["platform_fe_slope"]
    summary_lines.append(f"- {name}: platform-FE point slope={POINT_ASSOC[name]['platform_fe_slope']:.8g}; {fe['estimability_status']}; invalid={fe['invalid_replicates']}/{BOOTSTRAP_REPS}.")
write_text(OUT_ROOT/"STAGE07_SUMMARY.md","\n".join(summary_lines)+"\n")

manifest={
    "schema_version":1,"stage_id":STAGE_ID,"stage_name":STAGE_NAME,"bridge_version":BRIDGE_VERSION,
    "protocol_version":PROTOCOL_VERSION,"execution_utc":utc_now(),
    "upstream":{"mode":"EXACT_CANONICAL_STAGE05_TO_STAGE06_CHAIN","stage05_manifest_sha256":EXPECTED_STAGE05["manifest_sha256"],"stage05_checksum_sha256":EXPECTED_STAGE05["checksum_sha256"],"stage05_status_sha256":EXPECTED_STAGE05["status_sha256"],"prediction_sha256":PRED_IDENTITY,"corrected_h3_sha256":H3_IDENTITY["h3_results"],"stage06_manifest_sha256":EXPECTED_STAGE06["manifest_sha256"],"stage06_checksum_sha256":EXPECTED_STAGE06["checksum_sha256"],"stage06_status_sha256":EXPECTED_STAGE06["status_sha256"],"stage06_h4_sha256":S6_IDENTITY["h4_results"]},
    "scientific_contract":{"cues":CUE_FAMILIES,"platforms":PLATFORMS,"models":MODEL_KEYS,"seeds":SEEDS,
        "primary_budget":PRIMARY_BUDGET,"bootstrap_reps":BOOTSTRAP_REPS,"bootstrap_seed":BOOTSTRAP_SEED,
        "exploratory_secondary":True,"causal":False,"cue_specific_reranking":False,"posthoc_cell_deletion":False,"conditional_valid_only_inference":False,"loo_secondary_robustness":True,"loo_modifies_primary_bridge":False},
    "artifacts":{x:sha256_file(OUT_ROOT/x) for x in required},
    "summary_sha256":sha256_file(OUT_ROOT/"STAGE07_SUMMARY.md"),
}
payload=dict(manifest)
manifest["manifest_payload_sha256"]=hashlib.sha256(json.dumps(_json_builtin(payload),sort_keys=True,separators=(",",":"),ensure_ascii=False).encode("utf-8")).hexdigest()
write_json(OUT_ROOT/"REPRODUCIBILITY_MANIFEST.json",manifest)

entries=[]
for p in sorted(OUT_ROOT.rglob("*"),key=str):
    if p.is_file() and p.name not in {"STAGE07_SHA256SUMS.txt","STAGE_STATUS.json"}:
        entries.append((str(p.relative_to(OUT_ROOT)).replace("\\","/"),sha256_file(p)))
write_text(OUT_ROOT/"STAGE07_SHA256SUMS.txt","\n".join(f"{h}  {r}" for r,h in entries)+"\n")
set_status("PASS","Exploratory H3→H4 bridge completed with strict parity and estimability gates",{"output_root":str(OUT_ROOT),"bootstrap_seconds":BOOTSTRAP_SECONDS})
print("STAGE 07 COMPLETE — PASS",flush=True)


STAGE 07 STATUS: PASS — Exploratory H3→H4 bridge completed with strict parity and estimability gates


STAGE 07 COMPLETE — PASS
